# The Sum of Probabilities (Joint Events)

This notebook extends the sum rule to **joint events** — events that *can* happen at the same time. When events overlap, simply adding their probabilities double-counts the overlap, and the **inclusion-exclusion principle** corrects for it. It is based on the fifth video of Week 1 (Introduction to Probability).


## 1. Introduction

Say the probability of rain is 80% and the probability of wind is 70%. What is the probability of rain **or** wind? If you just add, you get $80\% + 70\% = 150\%$ — way too much, and completely wrong.

The problem: rain and wind can happen **at the same time**, so the events are not disjoint. In real-world situations events are often *not* disjoint, and we need to account for overlapping outcomes.

After completing this notebook you will be able to:

- Recognize joint (non-mutually-exclusive) events and why naive addition overcounts.
- Define the intersection $A \cap B$ of two events.
- Apply the inclusion-exclusion principle $P(A \cup B) = P(A) + P(B) - P(A \cap B)$.
- See the disjoint sum rule as the special case where the intersection is empty.


## 2. Theory

### The intersection of events

The **intersection** of events $A$ and $B$, written $A \cap B$, is the event that **both** happen at the same time — e.g. the kids who play soccer *and* basketball. In a Venn diagram it is the region where the two circles overlap.

### The inclusion-exclusion principle

For counting set sizes:

$$
|A \cup B| = |A| + |B| - |A \cap B|
$$

Adding $|A|$ and $|B|$ counts every element of the overlap **twice**, so we subtract it once. The same rule holds for probabilities:

$$
P(A \cup B) = P(A) + P(B) - P(A \cap B)
$$

### Disjoint events are a special case

If $A$ and $B$ are disjoint, the intersection is the empty set and $P(A \cap B) = 0$, so the formula reduces to the simple sum rule $P(A \cup B) = P(A) + P(B)$ from the previous notebook. Terminology:

| Events | Also called | Union formula |
|---|---|---|
| Disjoint | Mutually exclusive | $P(A) + P(B)$ |
| Joint | Non-mutually exclusive | $P(A) + P(B) - P(A \cap B)$ |


## 3. Worked Examples

### Example 1: Soccer or basketball, when kids can play both

The school no longer restricts kids to one sport. Out of 10 kids: **6 play soccer, 5 play basketball, and 3 play both**. How many play soccer or basketball?

Adding 6 + 5 = 11 is more than the whole school — the 3 kids who play both sports were counted twice. Subtract them once:

$$
|S \cup B| = 6 + 5 - 3 = 8
$$

(The remaining 2 kids play neither sport.)

### Example 2: The same problem with probabilities

$P(S) = 0.6$, $P(B) = 0.5$, and $P(S \cap B) = 0.3$. Then

$$
P(S \cup B) = P(S) + P(B) - P(S \cap B) = 0.6 + 0.5 - 0.3 = 0.8
$$

Without knowing the intersection, the problem cannot be solved — that extra piece of information is exactly what the naive sum is missing.

### Example 3: A sum of 7 or a difference of 1 (two dice)

- Event $A$: sum of 7 — 6 outcomes: $(1,6), (2,5), (3,4), (4,3), (5,2), (6,1)$.
- Event $B$: difference of 1 (either direction) — 10 outcomes.
- Intersection $A \cap B$: outcomes with sum 7 **and** difference 1 — exactly $(3,4)$ and $(4,3)$, so 2 outcomes.

These two outcomes would be double-counted, so:

$$
P(A \cup B) = \frac{6}{36} + \frac{10}{36} - \frac{2}{36} = \frac{14}{36} = \frac{7}{18}
$$


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

rng = np.random.default_rng(seed=42)


### Examples 1 and 2 in code: counts and probabilities


In [ ]:
total_kids = 10
soccer, basketball, both = 6, 5, 3

union_count = soccer + basketball - both
print(f"Kids playing soccer or basketball: {soccer} + {basketball} - {both} = {union_count}")
print(f"Kids playing neither: {total_kids - union_count}")

p_s, p_b, p_both = 0.6, 0.5, 0.3
p_union = p_s + p_b - p_both
print(f"P(S ∪ B) = {p_s} + {p_b} - {p_both} = {p_union:.1f}")


### Example 3 in code: sum of 7 or difference of 1

Enumerating the 36 outcomes shows exactly which two are double-counted, and confirms that inclusion-exclusion matches the direct count of the union.


In [ ]:
two_dice = list(product(range(1, 7), repeat=2))

sum7 = {p for p in two_dice if sum(p) == 7}
diff1 = {p for p in two_dice if abs(p[0] - p[1]) == 1}
overlap = sum7 & diff1
union = sum7 | diff1

print(f"|A| (sum 7)      = {len(sum7)}")
print(f"|B| (diff 1)     = {len(diff1)}")
print(f"|A ∩ B|          = {len(overlap)}  -> {sorted(overlap)}")
print(f"|A ∪ B| direct   = {len(union)}")
print(f"Inclusion-exclusion: {len(sum7)} + {len(diff1)} - {len(overlap)} = "
      f"{len(sum7) + len(diff1) - len(overlap)}")
print(f"P(A ∪ B) = 14/36 = {14/36:.4f}")


### What goes wrong without the correction

A simulation shows that the naive sum overestimates, while inclusion-exclusion matches the observed frequency.


In [ ]:
n_trials = 200_000
dice = rng.integers(1, 7, size=(n_trials, 2))
sums = dice.sum(axis=1)
diffs = np.abs(dice[:, 0] - dice[:, 1])

simulated = ((sums == 7) | (diffs == 1)).mean()
naive = 6 / 36 + 10 / 36
corrected = 6 / 36 + 10 / 36 - 2 / 36

print(f"Simulated P(sum 7 or diff 1):  {simulated:.4f}")
print(f"Naive sum (wrong):             {naive:.4f}")
print(f"Inclusion-exclusion (right):   {corrected:.4f}")


## 5. Visualizations

### Joint events in a Venn diagram

Now the circles **overlap**. Adding the two circle areas counts the overlap twice — the inclusion-exclusion formula subtracts it once.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

rectangle = plt.Rectangle((0, 0), 10, 6, facecolor="#d8f0d8",
                          edgecolor="green", linewidth=2)
ax.add_patch(rectangle)

soccer = plt.Circle((4, 3), 1.9, facecolor="#7fc97f", alpha=0.7,
                    edgecolor="darkgreen", linewidth=2)
basketball = plt.Circle((6.2, 3), 1.75, facecolor="#fdc086", alpha=0.7,
                        edgecolor="darkorange", linewidth=2)
ax.add_patch(soccer)
ax.add_patch(basketball)

ax.text(3.2, 3, "S: soccer\nP = 0.6", ha="center", va="center", fontsize=10)
ax.text(7.1, 3, "B: basketball\nP = 0.5", ha="center", va="center", fontsize=10)
ax.text(5.1, 3, "S ∩ B\n0.3", ha="center", va="center", fontsize=10)
ax.text(5, 0.4, "P(S ∪ B) = 0.6 + 0.5 - 0.3 = 0.8", ha="center", fontsize=11)
ax.text(0.2, 5.6, "Sample space: all kids (100%)", fontsize=10)

ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.5, 6.5)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Joint events: the overlap would be counted twice, so subtract it once")
plt.show()


### The two-dice sample space, with the double-counted cells

Sum of 7 in red, difference of 1 in yellow — and the two outcomes in **both** events, $(3,4)$ and $(4,3)$, in purple. The union has $6 + 10 - 2 = 14$ cells.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

for first in range(1, 7):
    for second in range(1, 7):
        in_sum7 = first + second == 7
        in_diff1 = abs(first - second) == 1
        if in_sum7 and in_diff1:
            color = "#b07aa1"   # intersection: counted twice by the naive sum
        elif in_sum7:
            color = "#e15759"
        elif in_diff1:
            color = "#f1ce63"
        else:
            color = "#d8e6f2"
        rect = plt.Rectangle((second - 1, 6 - first), 1, 1, facecolor=color,
                             edgecolor="white", linewidth=2)
        ax.add_patch(rect)
        ax.text(second - 0.5, 6 - first + 0.5, f"{first},{second}",
                ha="center", va="center", fontsize=9)

ax.set_xlim(0, 6)
ax.set_ylim(0, 6)
ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect("equal")
ax.set_title("Sum 7 (red), diff 1 (yellow), both (purple): 6 + 10 - 2 = 14 cells")
plt.show()


## 6. Exercises

### Basic

1. $P(A) = 0.5$, $P(B) = 0.4$, $P(A \cap B) = 0.2$. What is $P(A \cup B)$?
2. In a class of 20 students, 12 like math, 9 like physics, and 5 like both. How many like math or physics?

### Intermediate

3. You roll one die. Compute $P(\text{odd} \cup \text{five})$ with inclusion-exclusion. (This is the case where the naive sum rule from the previous notebook fails.)
4. You roll two dice. What is the probability that the sum is 8 **or** both dice show the same number?

### Advanced

5. You draw one card from a standard 52-card deck. What is the probability that it is a heart **or** a face card (jack, queen, king)? Verify by enumerating the deck in Python.


## 7. Solutions


In [ ]:
# Exercise 1
print("1.  P(A ∪ B) = 0.5 + 0.4 - 0.2 =", round(0.5 + 0.4 - 0.2, 4))

# Exercise 2
print("2.  |M ∪ P| = 12 + 9 - 5 =", 12 + 9 - 5)

# Exercise 3: odd = {1,3,5}, five = {5}, intersection = {5}
print("3.  P(odd ∪ 5) = 3/6 + 1/6 - 1/6 =", round(3 / 6, 4))

# Exercise 4: sum of 8 or doubles
two_dice = list(product(range(1, 7), repeat=2))
sum8 = {p for p in two_dice if sum(p) == 8}
doubles = {p for p in two_dice if p[0] == p[1]}
inter = sum8 & doubles          # only (4,4)
p = (len(sum8) + len(doubles) - len(inter)) / 36
print(f"4.  P = {len(sum8)}/36 + {len(doubles)}/36 - {len(inter)}/36 = "
      f"{len(sum8) + len(doubles) - len(inter)}/36 = {p:.4f}")

# Exercise 5: hearts or face cards
suits = ["hearts", "diamonds", "clubs", "spades"]
ranks = ["A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"]
deck = [(rank, suit) for suit in suits for rank in ranks]

hearts = {c for c in deck if c[1] == "hearts"}
faces = {c for c in deck if c[0] in ("J", "Q", "K")}
union = hearts | faces
p_rule = (len(hearts) + len(faces) - len(hearts & faces)) / len(deck)
print(f"5.  P = 13/52 + 12/52 - 3/52 = 22/52 = {p_rule:.4f} "
      f"(direct count: {len(union)}/52)")


## 8. Conclusion

- The **intersection** $A \cap B$ is the event that both $A$ and $B$ happen at the same time.
- For joint events, naive addition double-counts the intersection. The **inclusion-exclusion principle** fixes this:

$$
P(A \cup B) = P(A) + P(B) - P(A \cap B)
$$

- The disjoint sum rule is the special case $P(A \cap B) = 0$ — so this formula is the one that *always* works.
- To use it you need one extra piece of information compared to the disjoint case: the probability of the overlap.
